In [1]:
"""
Common setup for SAND ALS severity classification.
Shared by both SVM and RandomForest versions.
"""

import os, glob, time
import numpy as np, pandas as pd
import soundfile as sf, librosa
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# CONFIG
# -------------------------
SR_TARGET = 16000
FRAME_WIN = 0.020
FRAME_HOP = 0.010
Nw_default = 0.8
Nsh = 0.1
RANDOM_STATE = 42

DATASET_DIR = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training"
METADATA_PATH = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx"
print(f"✅ Using dataset: {DATASET_DIR}")

# -------------------------
# Audio helpers
# -------------------------
def load_audio(filename, sr=SR_TARGET):
    x, orig_sr = sf.read(filename)
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    if orig_sr != sr:
        x = librosa.resample(y=x.astype(np.float32), orig_sr=orig_sr, target_sr=sr)
    return x.astype(np.float32), sr

def compute_mfcc_36(x, sr=SR_TARGET):
    hop_length = int(round(FRAME_HOP * sr))
    win_length = int(round(FRAME_WIN * sr))
    mfcc = librosa.feature.mfcc(
        y=x, sr=sr, n_mfcc=13, n_fft=max(512, win_length * 2), hop_length=hop_length
    )
    mfcc12 = mfcc[1:13, :]
    delta = librosa.feature.delta(mfcc12)
    delta2 = librosa.feature.delta(mfcc12, order=2)
    return np.concatenate([mfcc12, delta, delta2], axis=0).T

def apply_cmvn(frames):
    mu = np.mean(frames, axis=0)
    sigma = np.std(frames, axis=0)
    sigma[sigma == 0] = 1.0
    return (frames - mu) / sigma

def suprasegmental_from_frames(frames, Nw=Nw_default, Nsh=Nsh):
    hop_s, win_s = FRAME_HOP, FRAME_WIN
    frames_per_window = 1 if Nw <= win_s else int(round((Nw - win_s) / hop_s)) + 1
    shift = max(1, int(round(Nsh / hop_s)))
    n_frames = frames.shape[0]
    supraseg_feats = []
    start = 0
    while start + frames_per_window <= n_frames:
        w = frames[start:start+frames_per_window]
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)],0))
        start += shift
    if not supraseg_feats:
        w = frames
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)],0))
    return np.array(supraseg_feats)

# -------------------------
# Dataset builder
# -------------------------
def build_dataset(root_dir, metadata_path):
    meta_df = pd.read_excel(metadata_path)
    meta_df.columns = [c.strip().lower() for c in meta_df.columns]
    id_col = next(c for c in meta_df.columns if "id" in c)
    class_col = next(c for c in meta_df.columns if "class" in c)
    id_to_label = {str(r[id_col]).strip().upper(): int(r[class_col]) for _, r in meta_df.iterrows()}

    entries = []
    for task_dir in sorted(glob.glob(os.path.join(root_dir, "*"))):
        if not os.path.isdir(task_dir): continue
        task_name = os.path.basename(task_dir)
        wavs = sorted(glob.glob(os.path.join(task_dir, "*.wav")) + glob.glob(os.path.join(task_dir, "*.WAV")))
        for w in wavs:
            base = os.path.splitext(os.path.basename(w))[0].strip().upper()
            subj_id = base.split("_")[0].replace("-", "").replace(" ", "")
            match = next((k for k in id_to_label if k.replace("-", "").replace(" ", "") == subj_id), None)
            if match:
                entries.append({
                    "subject_id": match,
                    "utterance_path": w,
                    "label": id_to_label[match],
                    "task": task_name,
                })
    print(f"✅ Found {len(entries)} utterances from {len(set(e['subject_id'] for e in entries))} subjects.")
    return entries

def extract_utterance_supraseg(utt_path, Nw=Nw_default):
    x, sr = load_audio(utt_path)
    frames = compute_mfcc_36(x, sr)
    frames = apply_cmvn(frames)
    return suprasegmental_from_frames(frames, Nw, Nsh)
from sklearn.svm import SVC

from sklearn.svm import SVC
from tqdm import tqdm
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split

from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

def evaluate_pipeline_rf(entries, Nw=0.8, n_folds=3):
    """Random Forest classifier with tqdm progress and logging."""
    if len(entries) == 0:
        raise ValueError("No entries found.")

    subjects = np.array(sorted(set(e["subject_id"] for e in entries)))
    labels = np.array([next(e["label"] for e in entries if e["subject_id"] == s) for s in subjects])
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    accs, f1m, f1w = [], [], []

    print(f"\n🚀 Starting RandomForest evaluation with {n_folds}-fold cross-validation")
    print(f"🧮 Total subjects: {len(subjects)} | Total utterances: {len(entries)}\n")

    for fold, (train_idx, test_idx) in enumerate(skf.split(subjects, labels), start=1):
        fold_start = time.time()
        print(f"\n===== 🌲 Fold {fold}/{n_folds} =====")

        train_subj, test_subj = set(subjects[train_idx]), set(subjects[test_idx])
        X_train, y_train, X_test, y_test = [], [], [], []

        # ---------- Feature Extraction ----------
        for e in tqdm(entries, desc=f"🎧 Extracting features (fold {fold})", unit="file"):
            try:
                supr = extract_utterance_supraseg(e["utterance_path"], Nw)
            except Exception as ex:
                print(f"⚠️ Skipped {e['utterance_path']}: {ex}")
                continue
            for s in supr:
                if e["subject_id"] in train_subj: X_train.append(s); y_train.append(e["label"])
                elif e["subject_id"] in test_subj: X_test.append(s); y_test.append(e["label"])

        X_train, y_train = np.asarray(X_train), np.asarray(y_train)
        X_test, y_test = np.asarray(X_test), np.asarray(y_test)
        print(f"✅ Feature extraction done: Train={X_train.shape}, Test={X_test.shape}")

        # ---------- Scaling ----------
        print("⚖️ Scaling features...")
        scaler = StandardScaler().fit(X_train)
        X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

        # ---------- Model Training ----------
        print("🌲 Training Random Forest model...")
        model = RandomForestClassifier(
            n_estimators=150,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        model.fit(X_train, y_train)

        # ---------- Evaluation ----------
        print("🔎 Evaluating on test set...")
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average="macro")
        f1_weighted = f1_score(y_test, y_pred, average="weighted")

        print(f"✅ Fold {fold} Results:")
        print(f"   Accuracy={acc:.3f} | F1_macro={f1_macro:.3f} | F1_weighted={f1_weighted:.3f}")
        print(classification_report(y_test, y_pred, digits=3))

        accs.append(acc)
        f1m.append(f1_macro)
        f1w.append(f1_weighted)

        print(f"⏱ Fold {fold} completed in {(time.time() - fold_start)/60:.2f} minutes")

    print("\n🔥 FINAL RANDOM FOREST RESULTS 🔥")
    print(f"Average Accuracy = {np.mean(accs):.3f}")
    print(f"Average F1 (Macro) = {np.mean(f1m):.3f}")
    print(f"Average F1 (Weighted) = {np.mean(f1w):.3f}")
    print("🏁 All folds completed successfully!")

# Run Random Forest pipeline
entries = build_dataset(DATASET_DIR, METADATA_PATH)
evaluate_pipeline_rf(entries, n_folds=3)


✅ Using dataset: /kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training
✅ Found 2176 utterances from 272 subjects.

🚀 Starting RandomForest evaluation with 3-fold cross-validation
🧮 Total subjects: 272 | Total utterances: 2176


===== 🌲 Fold 1/3 =====


🎧 Extracting features (fold 1): 100%|██████████| 2176/2176 [02:23<00:00, 15.21file/s]


✅ Feature extraction done: Train=(184487, 108), Test=(100591, 108)
⚖️ Scaling features...
🌲 Training Random Forest model...


KeyboardInterrupt: 

In [2]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com
!nvidia-smi


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
Wed Nov 12 14:56:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |   

In [ ]:
import os

root = "/kaggle/input/sandspgc"
for dirpath, dirnames, filenames in os.walk(root):
    print(dirpath)
    if filenames:
        print("   Files:", filenames[:5])


In [1]:
!pip install --upgrade --quiet \
    cuml-cu12 cudf-cu12 cupy-cuda12x \
    --extra-index-url=https://pypi.nvidia.com



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 64.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 82.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.4/159.4 kB 124.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 63.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 MB 35.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 152.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 183.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.1/923.1 kB 189.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.3/627.3 MB 49.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 166.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 48.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 72.5 M

In [ ]:
"""
SAND ALS Severity Prediction — GPU-Accelerated XGBoost Pipeline
---------------------------------------------------------------
✅ Uses GPU (NVIDIA T4 on Kaggle)
✅ Computes Accuracy + F1-macro + F1-weighted
✅ Saves per-fold models (.json)
✅ Clean tqdm progress per fold
✅ Kaggle-ready: works directly with /kaggle/input/sandspgc
"""

# -------------------------
# 📦 IMPORTS
# -------------------------
import os, glob, time
import numpy as np, pandas as pd
import soundfile as sf, librosa
from tqdm import tqdm
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# ⚙️ CONFIGURATION
# -------------------------
SR_TARGET = 16000
FRAME_WIN = 0.020
FRAME_HOP = 0.010
Nw_default = 0.8
Nsh = 0.1
RANDOM_STATE = 42

DATASET_DIR = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training"
METADATA_PATH = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx"

print(f"✅ Dataset directory: {DATASET_DIR}")
print(f"✅ Metadata file: {METADATA_PATH}")

# -------------------------
# 🎧 AUDIO HELPERS
# -------------------------
def load_audio(filename, sr=SR_TARGET):
    x, orig_sr = sf.read(filename)
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    if orig_sr != sr:
        x = librosa.resample(y=x.astype(np.float32), orig_sr=orig_sr, target_sr=sr)
    return x.astype(np.float32), sr

def compute_mfcc_36(x, sr=SR_TARGET):
    hop = int(round(FRAME_HOP * sr))
    win = int(round(FRAME_WIN * sr))
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13, n_fft=max(512, win*2), hop_length=hop)
    mfcc12 = mfcc[1:13, :]
    delta = librosa.feature.delta(mfcc12)
    delta2 = librosa.feature.delta(mfcc12, order=2)
    return np.concatenate([mfcc12, delta, delta2], axis=0).T

def apply_cmvn(frames):
    mu, sigma = np.mean(frames, axis=0), np.std(frames, axis=0)
    sigma[sigma == 0] = 1.0
    return (frames - mu) / sigma

def suprasegmental_from_frames(frames, Nw=Nw_default, Nsh=Nsh):
    hop_s, win_s = FRAME_HOP, FRAME_WIN
    frames_per_window = 1 if Nw <= win_s else int(round((Nw - win_s) / hop_s)) + 1
    shift = max(1, int(round(Nsh / hop_s)))
    supraseg_feats, start = [], 0
    while start + frames_per_window <= frames.shape[0]:
        w = frames[start:start+frames_per_window]
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)], 0))
        start += shift
    if not supraseg_feats:
        w = frames
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)], 0))
    return np.array(supraseg_feats)

# -------------------------
# 🗂️ DATASET BUILDER
# -------------------------
def build_dataset(root_dir, metadata_path):
    meta_df = pd.read_excel(metadata_path)
    meta_df.columns = [c.strip().lower() for c in meta_df.columns]
    id_col = next(c for c in meta_df.columns if "id" in c)
    class_col = next(c for c in meta_df.columns if "class" in c)
    id_to_label = {str(r[id_col]).strip().upper(): int(r[class_col]) for _, r in meta_df.iterrows()}

    entries = []
    for task_dir in sorted(glob.glob(os.path.join(root_dir, "*"))):
        if not os.path.isdir(task_dir): continue
        task_name = os.path.basename(task_dir)
        wavs = glob.glob(os.path.join(task_dir, "*.wav")) + glob.glob(os.path.join(task_dir, "*.WAV"))
        for w in wavs:
            base = os.path.splitext(os.path.basename(w))[0].strip().upper()
            subj_id = base.split("_")[0].replace("-", "").replace(" ", "")
            match = next((k for k in id_to_label if k.replace("-", "").replace(" ", "") == subj_id), None)
            if match:
                entries.append({"subject_id": match, "utterance_path": w, "label": id_to_label[match], "task": task_name})
    print(f"✅ Found {len(entries)} utterances from {len(set(e['subject_id'] for e in entries))} subjects.")
    return entries

# -------------------------
# 🎚️ FEATURE EXTRACTION
# -------------------------
def extract_utterance_supraseg(utt_path, Nw=Nw_default):
    x, sr = load_audio(utt_path)
    frames = compute_mfcc_36(x, sr)
    frames = apply_cmvn(frames)
    return suprasegmental_from_frames(frames, Nw, Nsh)

# -------------------------
# ⚡ GPU XGBOOST EVALUATION
# -------------------------
def evaluate_pipeline_xgb(entries, Nw=0.8, n_folds=3, save_models=True, output_csv=True):
    if not entries:
        raise ValueError("No entries found.")
    
    subjects = np.array(sorted(set(e["subject_id"] for e in entries)))
    labels = np.array([next(e["label"] for e in entries if e["subject_id"] == s) for s in subjects])
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    accs, f1m, f1w = [], [], []
    all_preds = []

    print(f"\n🚀 Starting GPU XGBoost ({n_folds}-fold cross-validation)\n")

    for fold, (train_idx, test_idx) in enumerate(skf.split(subjects, labels), start=1):
        fold_start = time.time()
        print(f"\n===== ⚡ Fold {fold}/{n_folds} =====")

        train_subj, test_subj = set(subjects[train_idx]), set(subjects[test_idx])
        X_train, y_train, X_test, y_test = [], [], [], []

        # ---------- tqdm progress (one bar per fold)
        for e in tqdm(entries, desc=f"🎧 Extracting features (fold {fold})", unit="utterance"):
            try:
                supr = extract_utterance_supraseg(e["utterance_path"], Nw)
                for s in supr:
                    if e["subject_id"] in train_subj:
                        X_train.append(s)
                        y_train.append(e["label"])
                    elif e["subject_id"] in test_subj:
                        X_test.append(s)
                        y_test.append(e["label"])
            except Exception as ex:
                print(f"⚠️ Skipped {e['utterance_path']}: {ex}")
                continue

        X_train, y_train = np.asarray(X_train), np.asarray(y_train)
        X_test, y_test = np.asarray(X_test), np.asarray(y_test)
        print(f"✅ Features extracted: Train={X_train.shape}, Test={X_test.shape}")

        # ---------- Scale ----------
        print("⚖️ Scaling features...")
        scaler = StandardScaler().fit(X_train)
        X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

        # ---------- Train GPU XGBoost ----------
        print("🌲 Training GPU XGBoost model...")
        model = xgb.XGBClassifier(
            n_estimators=350,
            max_depth=12,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method='gpu_hist',
            predictor='gpu_predictor',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        )
        model.fit(X_train, y_train)

        # ---------- Evaluate ----------
        print("🔎 Evaluating...")
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average="macro")
        f1_weighted = f1_score(y_test, y_pred, average="weighted")

        print(f"✅ Fold {fold}: Accuracy={acc:.3f} | F1_macro={f1_macro:.3f} | F1_weighted={f1_weighted:.3f}")
        print(classification_report(y_test, y_pred, digits=3))

        accs.append(acc)
        f1m.append(f1_macro)
        f1w.append(f1_weighted)

        if save_models:
            model.save_model(f"/kaggle/working/xgb_fold{fold}.json")
            print(f"💾 Saved model: /kaggle/working/xgb_fold{fold}.json")

        for sid, p in zip(sorted(test_subj), [np.bincount(y_pred).argmax()]*len(test_subj)):
            all_preds.append((sid, p))

        print(f"⏱ Fold completed in {(time.time()-fold_start)/60:.2f} minutes")

    print("\n🔥 FINAL GPU XGBOOST RESULTS 🔥")
    print(f"Average Accuracy={np.mean(accs):.3f}")
    print(f"Average F1 (Macro)={np.mean(f1m):.3f}")
    print(f"Average F1 (Weighted)={np.mean(f1w):.3f}")

    if output_csv:
        df_pred = pd.DataFrame(all_preds, columns=["subject_id", "predicted_class"]).drop_duplicates()
        csv_path = "/kaggle/working/fold_predictions.csv"
        df_pred.to_csv(csv_path, index=False)
        print(f"📁 Predictions saved to: {csv_path}")

    print("🏁 Completed successfully!\n")


# -------------------------
# 🚀 RUN PIPELINE
# -------------------------
entries = build_dataset(DATASET_DIR, METADATA_PATH)
evaluate_pipeline_xgb(entries, n_folds=3)


✅ Dataset directory: /kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training
✅ Metadata file: /kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx
✅ Found 2176 utterances from 272 subjects.

🚀 Starting GPU XGBoost (3-fold cross-validation)


===== ⚡ Fold 1/3 =====


🎧 Extracting features (fold 1):   4%|▍         | 94/2176 [00:20<01:56, 17.93utterance/s] 

In [1]:
!nvidia-smi


Wed Nov 12 15:12:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import xgboost as xgb
from xgboost import build_info

print("XGBoost version:", xgb.__version__)
print("Build Info:")
print(build_info())



XGBoost version: 2.0.3
Build Info:
{'BUILTIN_PREFETCH_PRESENT': True, 'CUDA_VERSION': [11, 8], 'DEBUG': False, 'GCC_VERSION': [9, 3, 1], 'MM_PREFETCH_PRESENT': True, 'NCCL_VERSION': [2, 19, 3], 'THRUST_VERSION': [1, 15, 1], 'USE_CUDA': True, 'USE_FEDERATED': True, 'USE_NCCL': True, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so'}


In [3]:
# === SAND ALS pipeline — XGBoost GPU (modern device='cuda' API) ===
# Paste into a Kaggle notebook cell and run.
# Requirements on Kaggle: GPU accelerator enabled. No extra pip installs required.

import os
import glob
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm
import xgboost as xgb
from xgboost import build_info
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# CONFIG
# -------------------------
SR_TARGET = 16000
FRAME_WIN = 0.020
FRAME_HOP = 0.010
Nw_default = 0.8
Nsh = 0.1
RANDOM_STATE = 42

DATASET_DIR = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training"
METADATA_PATH = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx"
MODEL_DIR = "/kaggle/working"
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Using dataset dir: {DATASET_DIR}")
print(f"Using metadata path: {METADATA_PATH}")

# -------------------------
# DIAGNOSTIC: XGBoost GPU & GPU presence
# -------------------------
print("\n--- Diagnostic: XGBoost & GPU ---")
try:
    info = build_info()
    print("XGBoost version:", xgb.__version__)
    print("Build info snippet:", {k: info.get(k) for k in ("USE_CUDA", "CUDA_VERSION", "USE_RMM")})
except Exception as ex:
    print("Could not read xgboost build_info():", ex)
# detect GPUs via cupy if available (not required)
gpu_available = False
try:
    import cupy as cp
    gpu_count = cp.cuda.runtime.getDeviceCount()
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    print(f"cupy detected GPUs: {gpu_count}; example name: {gpu_name}")
    gpu_available = gpu_count > 0
except Exception:
    # fallback: nvidia-smi check via os.system would be noisy; keep it simple
    print("cupy not available or couldn't detect GPU. Use !nvidia-smi in a shell cell to confirm.")
    # if build_info says USE_CUDA True, we still consider GPU usable
    try:
        if info.get("USE_CUDA", False):
            gpu_available = True
    except Exception:
        pass

if info.get("USE_CUDA", False) and gpu_available:
    print("✅ XGBoost built with CUDA and GPU appears available. Will use device='cuda'.")
else:
    print("⚠️ XGBoost GPU support or GPU availability not fully detected.")
    if info.get("USE_CUDA", False):
        print("   XGBoost reports USE_CUDA=True but GPU detection failed; you may still be able to use GPU.")
    else:
        print("   XGBoost appears CPU-only; the pipeline will fall back to CPU (device='cpu').")

# -------------------------
# AUDIO / FEATURE HELPERS
# -------------------------
def load_audio(filename, sr=SR_TARGET):
    x, orig_sr = sf.read(filename)
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    if orig_sr != sr:
        x = librosa.resample(y=x.astype(np.float32), orig_sr=orig_sr, target_sr=sr)
    return x.astype(np.float32), sr

def compute_mfcc_36(x, sr=SR_TARGET):
    hop = int(round(FRAME_HOP * sr))
    win = int(round(FRAME_WIN * sr))
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13, n_fft=max(512, win*2), hop_length=hop)
    mfcc12 = mfcc[1:13, :]
    delta = librosa.feature.delta(mfcc12)
    delta2 = librosa.feature.delta(mfcc12, order=2)
    return np.concatenate([mfcc12, delta, delta2], axis=0).T

def apply_cmvn(frames):
    mu = np.mean(frames, axis=0)
    sigma = np.std(frames, axis=0)
    sigma[sigma == 0] = 1.0
    return (frames - mu) / sigma

def suprasegmental_from_frames(frames, Nw=Nw_default, Nsh=Nsh):
    hop_s, win_s = FRAME_HOP, FRAME_WIN
    if Nw <= win_s:
        frames_per_window = 1
    else:
        frames_per_window = int(round((Nw - win_s) / hop_s)) + 1
    shift = max(1, int(round(Nsh / hop_s)))
    n_frames = frames.shape[0]
    supraseg_feats = []
    start = 0
    while start + frames_per_window <= n_frames:
        w = frames[start:start + frames_per_window, :]
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)], axis=0))
        start += shift
    if len(supraseg_feats) == 0:
        w = frames
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)], axis=0))
    return np.array(supraseg_feats)

# -------------------------
# DATASET BUILDER
# -------------------------
def build_dataset(root_dir, metadata_path):
    meta_df = pd.read_excel(metadata_path)
    meta_df.columns = [c.strip().lower() for c in meta_df.columns]
    id_col = next((c for c in meta_df.columns if "id" in c), None)
    class_col = next((c for c in meta_df.columns if "class" in c), None)
    if id_col is None or class_col is None:
        raise ValueError("Metadata must contain ID and Class columns.")
    id_to_label = {}
    for _, row in meta_df.iterrows():
        key = str(row[id_col]).strip().upper()
        try:
            lbl = int(row[class_col])
        except Exception:
            lbl = int(float(row[class_col]))
        id_to_label[key] = lbl-1

    entries = []
    for task_dir in sorted(glob.glob(os.path.join(root_dir, "*"))):
        if not os.path.isdir(task_dir):
            continue
        task_name = os.path.basename(task_dir)
        wavs = sorted(glob.glob(os.path.join(task_dir, "*.wav")) + glob.glob(os.path.join(task_dir, "*.WAV")))
        for w in wavs:
            base = os.path.splitext(os.path.basename(w))[0].strip().upper()
            subj_id = base.split("_")[0]
            subj_id_clean = subj_id.replace("-", "").replace(" ", "")
            match = None
            for k in id_to_label.keys():
                if k.replace("-", "").replace(" ", "") == subj_id_clean:
                    match = k
                    break
            if match is None:
                # skip files not in metadata
                continue
            entries.append({
                "subject_id": match,
                "utterance_path": w,
                "label": id_to_label[match],
                "task": task_name
            })
    print(f"Found {len(entries)} utterances from {len(set(e['subject_id'] for e in entries))} subjects.")
    return entries

# -------------------------
# FEATURE EXTRACTION (single call per .wav)
# -------------------------
def extract_utterance_supraseg(utt_path, Nw=Nw_default):
    x, sr = load_audio(utt_path)
    frames = compute_mfcc_36(x, sr)
    frames = apply_cmvn(frames)
    return suprasegmental_from_frames(frames, Nw, Nsh)

# -------------------------
# MAIN: GPU XGBoost training with tidy tqdm
# -------------------------
def evaluate_pipeline_xgb(entries, Nw=0.8, n_folds=3, save_models=True, output_csv=True, device_if_gpu='cuda'):
    if not entries:
        raise ValueError("No entries found.")

    # prepare subjects and labels (subject-level label)
    subjects = np.array(sorted(set(e["subject_id"] for e in entries)))
    labels = np.array([next(e["label"] for e in entries if e["subject_id"] == s) for s in subjects])
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    accs, f1m, f1w = [], [], []
    subject_pred_records = []  # (subject_id, pred_label) across folds

    # choose device
    use_device = 'cuda' if (info.get("USE_CUDA", False) and gpu_available) else 'cpu'
    print(f"Training device selected: {use_device}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(subjects, labels), start=1):
        fold_start = time.time()
        print(f"\n===== Fold {fold}/{n_folds} =====")
        train_subjects = set(subjects[train_idx])
        test_subjects = set(subjects[test_idx])

        X_train, y_train, X_test, y_test = [], [], [], []

        # single tqdm bar for overall feature extraction this fold
        for e in tqdm(entries, desc=f"Extracting features (fold {fold})", unit="utterance", ncols=100):
            try:
                supr = extract_utterance_supraseg(e["utterance_path"], Nw)
                if supr is None or supr.size == 0:
                    continue
                if e["subject_id"] in train_subjects:
                    for s in supr:
                        X_train.append(s); y_train.append(e["label"])
                elif e["subject_id"] in test_subjects:
                    for s in supr:
                        X_test.append(s); y_test.append(e["label"])
            except Exception as ex:
                # keep going if a file is bad
                print(f"⚠️ Skipping {e['utterance_path']}: {ex}")
                continue

        X_train = np.asarray(X_train)
        y_train = np.asarray(y_train)
        X_test = np.asarray(X_test)
        y_test = np.asarray(y_test)

        print(f"Features: Train={X_train.shape}, Test={X_test.shape}")

        # if no train/test frames (unlikely) skip
        if X_train.shape[0] == 0 or X_test.shape[0] == 0:
            print("⚠️ No frames for this fold — skipping.")
            continue

        # scale features (fit on train)
        scaler = StandardScaler().fit(X_train)
        X_train_s = scaler.transform(X_train)
        X_test_s = scaler.transform(X_test)

        # XGBoost parameters — use device='cuda' if available
        model = xgb.XGBClassifier(
            n_estimators=350,
            max_depth=12,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method='hist',   # modern API: hist + device='cuda'
            device=('cuda' if use_device == 'cuda' else 'cpu'),
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=1
        )

        print("Training XGBoost (this will use GPU if device='cuda')...")
        model.fit(X_train_s, y_train)

        print("Evaluating...")
        y_pred = model.predict(X_test_s)
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')

        print(f"Fold {fold} results — Accuracy: {acc:.4f}, F1-macro: {f1_macro:.4f}, F1-weighted: {f1_weighted:.4f}")
        print(classification_report(y_test, y_pred, digits=4))

        accs.append(acc); f1m.append(f1_macro); f1w.append(f1_weighted)

        # save model
        if save_models:
            model_path = os.path.join(MODEL_DIR, f"xgb_fold{fold}.json")
            try:
                model.save_model(model_path)
                print(f"Saved model to: {model_path}")
            except Exception as ex:
                print("Model saving failed:", ex)

        # compute per-subject majority vote predictions from frame-level preds
        # gather predictions per utterance subject and majority vote per subject
        subj_to_preds = defaultdict(list)
        # To map frame predictions back to subjects, iterate again through entries
        # But we can use y_test labels and equal-length y_pred assignments because frames were appended in same order
        # We'll reconstruct mapping by iterating over entries in test_subjects and re-extracting window counts
        # Simpler: iterate entries in same order and consume frames from y_test/y_pred arrays.
        idx_ptr = 0
        for e in entries:
            if e["subject_id"] in test_subjects:
                try:
                    supr = extract_utterance_supraseg(e["utterance_path"], Nw)
                    if supr is None: continue
                    n_frames = supr.shape[0]
                    if n_frames == 0:
                        continue
                    # slice corresponding y_pred frames
                    preds_slice = y_pred[idx_ptr: idx_ptr + n_frames]
                    if len(preds_slice) != n_frames:
                        # mismatch; fall back to majority of pred so far
                        pass
                    if len(preds_slice) > 0:
                        subj_to_preds[e["subject_id"]].extend(preds_slice.tolist())
                    idx_ptr += n_frames
                except Exception:
                    continue

        # majority vote per subject in this fold; store predictions
        for sid in subj_to_preds:
            votes = subj_to_preds[sid]
            if len(votes) == 0: continue
            most = Counter(votes).most_common(1)[0][0]
            subject_pred_records.append((sid, most))

        print(f"Fold {fold} time: {(time.time() - fold_start)/60:.2f} min")

    # final summary
    print("\n=== FINAL SUMMARY ===")
    if len(accs) > 0:
        print(f"Avg Accuracy: {np.mean(accs):.4f}")
        print(f"Avg F1-macro: {np.mean(f1m):.4f}")
        print(f"Avg F1-weighted: {np.mean(f1w):.4f}")
    else:
        print("No folds completed successfully.")

    # save subject-level predictions (deduplicate by taking majority across folds)
    if output_csv and len(subject_pred_records) > 0:
        df_preds = pd.DataFrame(subject_pred_records, columns=["subject_id", "pred"])
        # take majority across folds for same subject if present multiple times
        final_preds = df_preds.groupby("subject_id")["pred"].agg(lambda arr: int(Counter(arr).most_common(1)[0][0])).reset_index()
        csv_path = os.path.join(MODEL_DIR, "fold_predictions.csv")
        final_preds.to_csv(csv_path, index=False)
        print(f"Saved per-subject predictions to: {csv_path}")

# -------------------------
# RUN
# -------------------------
entries = build_dataset(DATASET_DIR, METADATA_PATH)
evaluate_pipeline_xgb(entries, n_folds=3, save_models=True, output_csv=True)


Using dataset dir: /kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training
Using metadata path: /kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx

--- Diagnostic: XGBoost & GPU ---
XGBoost version: 2.0.3
Build info snippet: {'USE_CUDA': True, 'CUDA_VERSION': [11, 8], 'USE_RMM': False}
cupy detected GPUs: 2; example name: Tesla T4
✅ XGBoost built with CUDA and GPU appears available. Will use device='cuda'.
Found 2176 utterances from 272 subjects.
Training device selected: cuda

===== Fold 1/3 =====


Extracting features (fold 1): 100%|██████████████████████| 2176/2176 [01:21<00:00, 26.83utterance/s]


Features: Train=(184487, 108), Test=(100591, 108)
Training XGBoost (this will use GPU if device='cuda')...
Evaluating...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [18:29:39] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Fold 1 results — Accuracy: 0.4154, F1-macro: 0.2154, F1-weighted: 0.3753
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000      1436
           1     0.5066    0.0259    0.0493      4434
           2     0.3362    0.1054    0.1605     18771
           3     0.3171    0.3120    0.3145     31280
           4     0.4700    0.6701    0.5525     44670

    accuracy                         0.4154    100591
   macro avg     0.3260    0.2227    0.2154    100591
weighted avg     0.3924    0.4154    0.3753    100591

Saved model to: /kaggle/working/xgb_fold1.json
Fold 1 time: 3.78 min

===== Fold 2/3 =====


Extracting features (fold 2): 100%|██████████████████████| 2176/2176 [01:17<00:00, 28.09utterance/s]


Features: Train=(189532, 108), Test=(95546, 108)
Training XGBoost (this will use GPU if device='cuda')...
Evaluating...
Fold 2 results — Accuracy: 0.4017, F1-macro: 0.2070, F1-weighted: 0.3677
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       515
           1     0.1458    0.0051    0.0099      5446
           2     0.2661    0.1239    0.1691     16564
           3     0.3447    0.3212    0.3325     31106
           4     0.4486    0.6277    0.5232     41915

    accuracy                         0.4017     95546
   macro avg     0.2410    0.2156    0.2070     95546
weighted avg     0.3634    0.4017    0.3677     95546

Saved model to: /kaggle/working/xgb_fold2.json
Fold 2 time: 3.73 min

===== Fold 3/3 =====


Extracting features (fold 3): 100%|██████████████████████| 2176/2176 [01:16<00:00, 28.30utterance/s]


Features: Train=(196137, 108), Test=(88941, 108)
Training XGBoost (this will use GPU if device='cuda')...
Evaluating...
Fold 3 results — Accuracy: 0.4123, F1-macro: 0.2150, F1-weighted: 0.3588
              precision    recall  f1-score   support

           0     0.7500    0.0068    0.0135       441
           1     0.4280    0.0317    0.0590      3375
           2     0.3413    0.0992    0.1537     17510
           3     0.3970    0.2473    0.3048     32465
           4     0.4228    0.7622    0.5439     35150

    accuracy                         0.4123     88941
   macro avg     0.4678    0.2294    0.2150     88941
weighted avg     0.3992    0.4123    0.3588     88941

Saved model to: /kaggle/working/xgb_fold3.json
Fold 3 time: 3.71 min

=== FINAL SUMMARY ===
Avg Accuracy: 0.4098
Avg F1-macro: 0.2124
Avg F1-weighted: 0.3672
Saved per-subject predictions to: /kaggle/working/fold_predictions.csv


In [1]:
"""
Kaggle ALS Severity Classification (GPU-Optimized DNN Pipeline)
---------------------------------------------------------------
✔ DNN-only version (TensorFlow GPU)
✔ Handles multiple GPUs in Kaggle
✔ Suppresses TensorFlow/Protobuf/Plugin warnings
✔ MFCC + suprasegmental stats → DNN classifier
✔ 3-fold CV, Accuracy + F1
✔ Saves best models to /kaggle/working/
"""

# -------------------------------
# 1️⃣ Environment setup (suppress noisy logs)
# -------------------------------
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Optional: force single GPU (uncomment if memory conflict)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# -------------------------------
# 2️⃣ Imports
# -------------------------------
import glob
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# -------------------------------
# 3️⃣ GPU setup (handles multi-GPU Kaggle environment)
# -------------------------------
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise SystemError("❌ No GPU detected! Go to Settings → Accelerator → GPU.")
else:
    print(f"✅ GPUs detected: {[g.name for g in gpus]}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print("⚠️ Skipping memory growth:", e)

# -------------------------------
# 4️⃣ Config
# -------------------------------
SR_TARGET = 16000
FRAME_WIN = 0.020
FRAME_HOP = 0.010
Nw_default = 0.8
Nsh = 0.1
RANDOM_STATE = 42

DATASET_DIR = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training"
METADATA_PATH = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx"

print(f"📁 Dataset: {DATASET_DIR}")
print(f"📄 Metadata: {METADATA_PATH}")

# -------------------------------
# 5️⃣ Audio + Feature Functions
# -------------------------------
def load_audio(filename, sr=SR_TARGET):
    x, orig_sr = sf.read(filename)
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    if orig_sr != sr:
        x = librosa.resample(y=x.astype(np.float32), orig_sr=orig_sr, target_sr=sr)
    return x.astype(np.float32), sr

def compute_mfcc_36(x, sr=SR_TARGET):
    hop = int(round(FRAME_HOP * sr))
    win = int(round(FRAME_WIN * sr))
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13, n_fft=max(512, win * 2), hop_length=hop)
    mfcc12 = mfcc[1:13, :]
    delta = librosa.feature.delta(mfcc12)
    delta2 = librosa.feature.delta(mfcc12, order=2)
    return np.concatenate([mfcc12, delta, delta2], axis=0).T

def apply_cmvn(frames):
    mu, sigma = np.mean(frames, 0), np.std(frames, 0)
    sigma[sigma == 0] = 1.0
    return (frames - mu) / sigma

def suprasegmental_from_frames(frames, Nw=Nw_default, Nsh=Nsh):
    hop_s, win_s = FRAME_HOP, FRAME_WIN
    frames_per_window = int(round((Nw - win_s) / hop_s)) + 1 if Nw > win_s else 1
    shift = int(round(Nsh / hop_s))
    supraseg_feats, start = [], 0
    while start + frames_per_window <= frames.shape[0]:
        w = frames[start:start + frames_per_window]
        mean, med, sd = np.mean(w, 0), np.median(w, 0), np.std(w, 0)
        supraseg_feats.append(np.concatenate([mean, med, sd], 0))
        start += shift
    if not supraseg_feats:
        w = frames
        supraseg_feats.append(np.concatenate([np.mean(w, 0), np.median(w, 0), np.std(w, 0)], 0))
    return np.array(supraseg_feats)

def extract_utterance_supraseg(utt_path, Nw=Nw_default):
    x, sr = load_audio(utt_path)
    frames = compute_mfcc_36(x, sr)
    frames = apply_cmvn(frames)
    return suprasegmental_from_frames(frames, Nw, Nsh)

# -------------------------------
# 6️⃣ Dataset Builder
# -------------------------------
def build_dataset(root_dir, metadata_path):
    meta = pd.read_excel(metadata_path)
    meta.columns = [c.strip().lower() for c in meta.columns]
    id_col = next(c for c in meta.columns if "id" in c)
    class_col = next(c for c in meta.columns if "class" in c)
    id_to_label = {str(r[id_col]).strip().upper(): int(r[class_col]) - 1 for _, r in meta.iterrows()}

    entries = []
    for task_dir in sorted(glob.glob(os.path.join(root_dir, "*"))):
        if not os.path.isdir(task_dir): continue
        for w in glob.glob(os.path.join(task_dir, "*.wav")):
            base = os.path.splitext(os.path.basename(w))[0].strip().upper()
            subj_id = base.split("_")[0]
            subj_id_clean = subj_id.replace("-", "").replace(" ", "")
            match = next((k for k in id_to_label if k.replace("-", "").replace(" ", "") == subj_id_clean), None)
            if match:
                entries.append({
                    "subject_id": match,
                    "utterance_path": w,
                    "label": id_to_label[match]
                })
    print(f"✅ Found {len(entries)} utterances from {len(set(e['subject_id'] for e in entries))} subjects.")
    return entries

# -------------------------------
# 7️⃣ DNN Model
# -------------------------------
def build_dnn_model(input_dim, n_classes=5):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# -------------------------------
# 8️⃣ Evaluation (GPU DNN)
# -------------------------------
def evaluate_pipeline(entries, Nw=0.8, n_folds=3):
    subjects = np.array(sorted(set(e["subject_id"] for e in entries)))
    labels = np.array([next(e["label"] for e in entries if e["subject_id"] == s) for s in subjects])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    accs, f1s = [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(subjects, labels), 1):
        print(f"\n===== Fold {fold}/{n_folds} =====")
        train_subj, test_subj = set(subjects[tr_idx]), set(subjects[te_idx])

        X_train, y_train, X_test, y_test = [], [], [], []
        for e in tqdm(entries, desc=f"Extracting features (fold {fold})", ncols=90):
            try:
                supr = extract_utterance_supraseg(e["utterance_path"], Nw)
            except Exception as ex:
                print(f"⚠️ {e['utterance_path']} failed: {ex}")
                continue
            for s in supr:
                if e["subject_id"] in train_subj:
                    X_train.append(s); y_train.append(e["label"])
                elif e["subject_id"] in test_subj:
                    X_test.append(s); y_test.append(e["label"])

        X_train, y_train, X_test, y_test = map(np.array, (X_train, y_train, X_test, y_test))
        print(f"Train {X_train.shape}, Test {X_test.shape}")

        sc = StandardScaler().fit(X_train)
        X_train, X_test = sc.transform(X_train), sc.transform(X_test)

        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE)

        dnn = build_dnn_model(X_tr.shape[1])
        es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        ckpt_path = f"/kaggle/working/fold_{fold}_best.keras"
        mc = callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor='val_loss', mode='min')

        print("🚀 Training on GPU...")
        dnn.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                epochs=50, batch_size=128, verbose=1,
                callbacks=[es, mc])

        y_pred = np.argmax(dnn.predict(X_test, verbose=0), axis=1)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        print(f"✅ Fold {fold}: Accuracy={acc:.4f}, F1-macro={f1:.4f}")
        print(classification_report(y_test, y_pred, digits=3))
        accs.append(acc)
        f1s.append(f1)

    print("\n🔥 Final Results 🔥")
    print(f"Mean Accuracy: {np.mean(accs):.4f}")
    print(f"Mean F1-macro: {np.mean(f1s):.4f}")

# -------------------------------
# 9️⃣ Run
# -------------------------------
entries = build_dataset(DATASET_DIR, METADATA_PATH)
evaluate_pipeline(entries, Nw=0.8, n_folds=3)


E0000 00:00:1762973834.312744      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762973834.437661      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

1226/1226 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.3685 - loss: 1.4656 - val_accuracy: 0.4661 - val_loss: 1.1299
Epoch 2/50
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4585 - loss: 1.1468 - val_accuracy: 0.4955 - val_loss: 1.0889
Epoch 3/50
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4812 - loss: 1.1074 - val_accuracy: 0.5182 - val_loss: 1.0495
Epoch 4/50
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5035 - loss: 1.0745 - val_accuracy: 0.5308 - val_loss: 1.0181
Epoch 5/50
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5180 - loss: 1.0491 - val_accuracy: 0.5510 - val_loss: 0.9911
Epoch 6/50
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5301 - loss: 1.0272 - val_accuracy: 0.5704 - val_loss: 0.9681
Epoch 7/50
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5414 - loss: 1.0065 - val_accuracy: 0.5876 - val_loss: 0.9427
Epoch 8/50
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5537 - loss: 0.9898 - val_accura

Extracting features (fold 2): 100%|███████████████████| 2176/2176 [01:25<00:00, 25.57it/s]


Train (189532, 108), Test (95546, 108)
🚀 Training on GPU...
Epoch 1/50
1259/1259 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.3748 - loss: 1.4769 - val_accuracy: 0.4731 - val_loss: 1.1314
Epoch 2/50
1259/1259 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4639 - loss: 1.1474 - val_accuracy: 0.5046 - val_loss: 1.0801
Epoch 3/50
1259/1259 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4877 - loss: 1.1040 - val_accuracy: 0.5303 - val_loss: 1.0341
Epoch 4/50
1259/1259 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5074 - loss: 1.0691 - val_accuracy: 0.5486 - val_loss: 0.9985
Epoch 5/50
1259/1259 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5235 - loss: 1.0406 - val_accuracy: 0.5635 - val_loss: 0.9764
Epoch 6/50
1259/1259 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5365 - loss: 1.0163 - val_accuracy: 0.5848 - val_loss: 0.9428
Epoch 7/50
1259/1259 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5470 - loss: 0.9988 - val_accuracy: 0.5969 - val_loss: 0.9248
Epoch 8/50
1259/1259 ━━━━━━━━

Extracting features (fold 3): 100%|███████████████████| 2176/2176 [01:25<00:00, 25.46it/s]


Train (196137, 108), Test (88941, 108)
🚀 Training on GPU...
Epoch 1/50
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.3792 - loss: 1.4761 - val_accuracy: 0.4728 - val_loss: 1.1461
Epoch 2/50
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.4631 - loss: 1.1628 - val_accuracy: 0.4892 - val_loss: 1.1038
Epoch 3/50
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4818 - loss: 1.1278 - val_accuracy: 0.5202 - val_loss: 1.0652
Epoch 4/50
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5007 - loss: 1.0951 - val_accuracy: 0.5370 - val_loss: 1.0332
Epoch 5/50
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5133 - loss: 1.0687 - val_accuracy: 0.5546 - val_loss: 1.0046
Epoch 6/50
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5294 - loss: 1.0442 - val_accuracy: 0.5684 - val_loss: 0.9817
Epoch 7/50
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5384 - loss: 1.0284 - val_accuracy: 0.5795 - val_loss: 0.9603
Epoch 8/50
1303/1303 ━━━━━━━━

In [4]:
# Combined pipeline: suprasegmental MFCC + spectral (mel) statistical features -> PyTorch DNN
# -------------------------------
# 0️⃣ Note: paths are unchanged from your first script (Kaggle).
# -------------------------------

# -------------------------------
# 1️⃣ Imports & Setup
# -------------------------------
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from scipy.stats import skew, kurtosis
from scipy.fftpack import dct

import librosa
import soundfile as sf

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# -------------------------------
# 2️⃣ GPU setup
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device detected: {device}")
if device.type != "cuda":
    raise SystemError("❌ GPU not detected. Please enable GPU in Kaggle runtime settings.")

# -------------------------------
# 3️⃣ Configuration
# -------------------------------
SR_TARGET = 16000
FRAME_WIN = 0.020
FRAME_HOP = 0.010
Nw_default = 0.8
Nsh = 0.1
BATCH_SIZE = 128
EPOCHS = 40
LR = 1e-3
RANDOM_STATE = 42

DATASET_DIR = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training"
METADATA_PATH = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx"

# -------------------------------
# 4️⃣ Audio basic helpers
# -------------------------------
def load_audio(filename, sr=SR_TARGET):
    x, orig_sr = sf.read(filename)
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    if orig_sr != sr:
        x = librosa.resample(y=x.astype(np.float32), orig_sr=orig_sr, target_sr=sr)
    return x.astype(np.float32), sr

# -------------------------------
# 5️⃣ MFCC suprasegmental features (as before)
# -------------------------------
def compute_mfcc_36(x, sr=SR_TARGET):
    hop = int(round(FRAME_HOP * sr))
    win = int(round(FRAME_WIN * sr))
    # 13 MFCCs (0..12), we keep 1..12 for the 12-dim base then deltas
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13, n_fft=max(512, win * 2), hop_length=hop)
    mfcc12 = mfcc[1:13, :]  # shape (12, frames)
    delta = librosa.feature.delta(mfcc12)
    delta2 = librosa.feature.delta(mfcc12, order=2)
    return np.concatenate([mfcc12, delta, delta2], axis=0).T  # (frames, 36)

def apply_cmvn(frames):
    mu, sigma = np.mean(frames, 0), np.std(frames, 0)
    sigma[sigma == 0] = 1.0
    return (frames - mu) / sigma

def suprasegmental_from_frames(frames, Nw=Nw_default, Nsh=Nsh):
    hop_s, win_s = FRAME_HOP, FRAME_WIN
    frames_per_window = int(round((Nw - win_s) / hop_s)) + 1 if Nw > win_s else 1
    shift = int(round(Nsh / hop_s))
    supraseg_feats, start = [], 0
    while start + frames_per_window <= frames.shape[0]:
        w = frames[start:start + frames_per_window]
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)],0))
        start += shift
    if not supraseg_feats:
        w = frames
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)],0))
    return np.array(supraseg_feats)  # (segments, 36*3 = 108)

def extract_utterance_supraseg(utt_path, Nw=Nw_default):
    x, sr = load_audio(utt_path)
    frames = compute_mfcc_36(x, sr)
    frames = apply_cmvn(frames)
    return suprasegmental_from_frames(frames, Nw, Nsh)

# -------------------------------
# 6️⃣ Spectral / Mel statistical features (adapted)
# -------------------------------
def extract_mel_features(mel, sr_mel=128):
    """
    mel: ndarray (n_mels, n_frames) - magnitude or intensity (not log) recommended
    returns: 1D vector of aggregated statistics
    """
    eps = 1e-10
    mel = np.maximum(np.asarray(mel, dtype=float), 0.0)
    n_mels, n_frames = mel.shape

    freqs = np.arange(n_mels, dtype=float) + 1.0
    mag = mel + eps

    # Spectral centroid & bandwidth
    centroid = (freqs[:, None] * mag).sum(axis=0) / (mag.sum(axis=0) + eps)
    bandwidth = np.sqrt(((freqs[:, None] - centroid[None, :])**2 * mag).sum(axis=0) / (mag.sum(axis=0) + eps))

    # Spectral rolloff (85%)
    cumsum = np.cumsum(mag, axis=0)
    total_energy = cumsum[-1, :] + eps
    rolloff = np.array([np.searchsorted(cumsum[:, t], 0.85 * total_energy[t]) for t in range(n_frames)])

    # Spectral flatness
    geo_mean = np.exp(np.mean(np.log(mag), axis=0))
    arith_mean = np.mean(mag, axis=0)
    flatness = geo_mean / (arith_mean + eps)

    # Spectral flux (frame diff)
    mag_norm = mag / (arith_mean[None, :] + eps)
    flux = np.sqrt(np.diff(mag_norm, axis=1)**2).sum(axis=0)
    flux = np.concatenate(([0.0], flux))

    # Spectral entropy
    p = mag / (mag.sum(axis=0, keepdims=True) + eps)
    entropy = - (p * np.log2(p + eps)).sum(axis=0) / np.log2(n_mels + eps)

    # Band energies (low/mid/high thirds)
    n1, n2 = n_mels // 3, 2 * n_mels // 3
    low_energy = mag[:n1, :].sum(axis=0)
    mid_energy = mag[n1:n2, :].sum(axis=0)
    high_energy = mag[n2:, :].sum(axis=0)

    # MFCC-like via DCT on log-mel
    log_mel = np.log(mag)
    mfccs = dct(log_mel, type=2, axis=0, norm='ortho')[:13, :]

    # Band contrast
    n_bands = 6
    band_edges = np.linspace(0, n_mels, n_bands + 1, dtype=int)
    contrasts = []
    for b in range(n_bands):
        band = mag[band_edges[b]:band_edges[b + 1], :]
        contrasts.append((band.max(axis=0) - band.min(axis=0)) if band.size > 0 else np.zeros(n_frames))
    contrasts = np.vstack(contrasts) if contrasts else np.zeros((0, n_frames))

    def stats_vec(v):
        v = np.asarray(v, dtype=float)
        if v.size == 0:
            return np.zeros(7, dtype=float)
        return np.array([
            np.nanmean(v), np.nanstd(v),
            skew(v, bias=False) if v.size > 2 else 0.0,
            kurtosis(v, bias=False) if v.size > 3 else 0.0,
            np.nanmedian(v),
            np.nanpercentile(v, 25.0),
            np.nanpercentile(v, 75.0)
        ])

    feats = []
    for vec in [centroid, bandwidth, rolloff, flatness, flux, entropy, low_energy, mid_energy, high_energy]:
        feats.append(stats_vec(vec))
    feats.append(stats_vec(np.mean(mag, axis=1)))
    feats.append(stats_vec(np.std(mag, axis=1)))
    for b in range(contrasts.shape[0]):
        feats.append(stats_vec(contrasts[b, :]))
    feats.append(np.hstack([np.mean(mfccs, axis=1), np.std(mfccs, axis=1)]))
    feats.append(np.array([np.nanmean(mag), np.nanstd(mag), np.nanmax(mag), np.nanmin(mag)]))

    return np.nan_to_num(np.hstack(feats))

def compute_mel_stats_from_audio(x, sr=SR_TARGET, n_mels=128, hop_length=None, n_fft=None):
    hop_length = hop_length or int(round(FRAME_HOP * sr))
    n_fft = n_fft or max(512, int(round(FRAME_WIN * sr)) * 2)
    S = librosa.feature.melspectrogram(y=x, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
    # Using magnitude spectrogram (not log) for the stats extractor which handles log internally where needed
    return extract_mel_features(S, sr_mel=n_mels)

# -------------------------------
# 7️⃣ Metadata loading (Age, Sex if available)
# -------------------------------
meta_df = pd.read_excel(METADATA_PATH)
meta_df.columns = [c.strip() for c in meta_df.columns]
# normalize column names to find ID, Class, Age, Sex
cols_low = [c.lower() for c in meta_df.columns]
id_col = next((c for c in meta_df.columns if 'id' in c.lower()), None)
class_col = next((c for c in meta_df.columns if 'class' in c.lower()), None)
age_col = next((c for c in meta_df.columns if 'age' in c.lower()), None)
sex_col = next((c for c in meta_df.columns if 'sex' in c.lower()), None)

# Prepare id -> row mapping
meta_df[id_col] = meta_df[id_col].astype(str).str.strip().str.upper()
meta_map = meta_df.set_index(id_col).to_dict(orient='index')

# Also create id->label
id_to_label = {str(r[id_col]).strip().upper(): int(r[class_col]) - 1 for _, r in meta_df.iterrows()}

# -------------------------------
# 8️⃣ Build dataset (gather file paths + labels)
# -------------------------------
def build_dataset(root_dir, id_to_label_map):
    entries = []
    for folder, _, files in os.walk(root_dir):
        for f in files:
            if not f.lower().endswith(".wav"):
                continue
            subj = f.split("_")[0].strip().upper()
            subj_clean = subj.replace("-", "").replace(" ", "")
            # try to match against id keys (some ids contain hyphens/spaces)
            match = next((k for k in id_to_label_map if k.replace("-", "").replace(" ", "") == subj_clean), None)
            if match:
                entries.append({
                    "subject_id": match,
                    "utterance_path": os.path.join(folder, f),
                    "label": id_to_label_map[match]
                })
    print(f"✅ Found {len(entries)} utterances from {len(set(e['subject_id'] for e in entries))} subjects.")
    return entries

# -------------------------------
# 9️⃣ Feature-level augmentation (minority classes)
# -------------------------------
def feature_augment(X, y, noise_std=0.01, random_state=42):
    rng = np.random.default_rng(random_state)
    X_aug, y_aug = [X], [y]
    unique, counts = np.unique(y, return_counts=True)
    max_count = np.max(counts)

    for cls, count in zip(unique, counts):
        if count < max_count:
            mask = (y == cls)
            n_needed = max_count - count
            X_minority = X[mask]
            if len(X_minority) == 0:
                continue
            idx = rng.choice(len(X_minority), n_needed, replace=True)
            X_new = X_minority[idx] + rng.normal(0, noise_std, X_minority[idx].shape)
            y_new = np.full(n_needed, cls)
            X_aug.append(X_new)
            y_aug.append(y_new)

    X_bal = np.vstack(X_aug)
    y_bal = np.hstack(y_aug)
    return X_bal, y_bal

# -------------------------------
# 10️⃣ Simple NumPy oversampler (keeps as extra option)
# -------------------------------
def random_oversample(X, y, random_state=42):
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y, return_counts=True)
    max_count = counts.max()
    X_res, y_res = [], []
    for cls, count in zip(classes, counts):
        mask = y == cls
        X_cls = X[mask]
        y_cls = y[mask]
        n_needed = max_count - count
        if n_needed > 0:
            idx = rng.choice(len(X_cls), n_needed, replace=True)
            X_extra = X_cls[idx]
            y_extra = np.full(n_needed, cls)
            X_cls = np.vstack([X_cls, X_extra])
            y_cls = np.hstack([y_cls, y_extra])
        X_res.append(X_cls)
        y_res.append(y_cls)
    return np.vstack(X_res), np.hstack(y_res)

# -------------------------------
# 11️⃣ DNN Model Definition
# -------------------------------
class DNNModel(nn.Module):
    def __init__(self, input_dim, n_classes=5):
        super(DNNModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    def forward(self, x):
        return self.net(x)

# -------------------------------
# 12️⃣ Training + Evaluation helpers
# -------------------------------
def train_one_fold(model, train_loader, val_loader, criterion, optimizer, epochs=EPOCHS):
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            if len(val_loader) > 0:
                val_loss = sum(criterion(model(Xv.to(device)), yv.to(device)).item() for Xv, yv in val_loader) / len(val_loader)
            else:
                val_loss = 0.0
        print(f"Epoch {epoch+1}/{epochs} | TrainLoss={total_loss/len(train_loader):.4f} | ValLoss={val_loss:.4f}")

# -------------------------------
# 13️⃣ Main evaluation pipeline (k-fold subject-wise)
# -------------------------------
def evaluate_pipeline(entries, Nw=0.8, n_folds=3, use_feature_augment=True, tile_mel_to_segments=True):
    # Build subject list and labels (subject-level stratify)
    subjects = np.array(sorted(set(e["subject_id"] for e in entries)))
    labels = np.array([next(e["label"] for e in entries if e["subject_id"] == s) for s in subjects])
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    fold_accs, fold_f1s = [], []
    total_conf_mat = None

    for fold, (train_idx, test_idx) in enumerate(skf.split(subjects, labels), 1):
        print(f"\n===== Fold {fold}/{n_folds} =====")
        train_subj, test_subj = set(subjects[train_idx]), set(subjects[test_idx])

        X_train, y_train, X_test, y_test = [], [], [], []

        # Extract features for each utterance, append mel stats (per utterance) to each supraseg segment
        desc = f"Extracting features (fold {fold})"
        for e in tqdm(entries, desc=desc, ncols=100):
            try:
                supr = extract_utterance_supraseg(e["utterance_path"], Nw)
            except Exception as exc:
                # skip if audio read fails
                continue
            # load full audio to compute mel stats
            try:
                x, sr = load_audio(e["utterance_path"])
                mel_feats = compute_mel_stats_from_audio(x, sr=sr, n_mels=128)
            except Exception:
                mel_feats = np.zeros( ( (13+13) + 7* (9 + 2) ) ) if True else np.zeros(100)  # fallback (not precise) - but we'll use zeros if fail
                mel_feats = np.zeros(256)  # fallback safe size; next scaling may error if sizes mismatch; but usually compute_mel_stats works

            # optional metadata features (Age, Sex)
            meta_feats = []
            subj_meta = meta_map.get(e["subject_id"], None)
            if subj_meta is not None:
                if age_col and age_col in subj_meta and pd.notna(subj_meta[age_col]):
                    meta_feats.append(float(subj_meta[age_col]))
                else:
                    meta_feats.append(0.0)
                if sex_col and sex_col in subj_meta and pd.notna(subj_meta[sex_col]):
                    sex_val = subj_meta[sex_col]
                    # map sex (M/F) to 0/1 if necessary
                    if isinstance(sex_val, str):
                        sex_val = 0.0 if sex_val.upper().startswith('M') else 1.0
                    meta_feats.append(float(sex_val))
                else:
                    meta_feats.append(0.0)
            else:
                meta_feats = [0.0, 0.0]

            # Combine supraseg segments with mel and meta: repeat/tile mel/meta if needed
            for segment in supr:
                if tile_mel_to_segments:
                    combined = np.hstack([segment, mel_feats, np.array(meta_feats)])
                else:
                    # alternate design: use segment + mel stats only once per utterance (not used here)
                    combined = np.hstack([segment, mel_feats, np.array(meta_feats)])
                if e["subject_id"] in train_subj:
                    X_train.append(combined); y_train.append(e["label"])
                elif e["subject_id"] in test_subj:
                    X_test.append(combined); y_test.append(e["label"])

        # ensure arrays
        X_train, y_train, X_test, y_test = map(np.array, (X_train, y_train, X_test, y_test))
        print("Raw sizes ->", X_train.shape, y_train.shape, X_test.shape, y_test.shape)
        if X_train.size == 0 or X_test.size == 0:
            print("Warning: no train/test data for this fold; skipping.")
            continue

        # Standard scaling
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

        # Feature-level augmentation (small noise) then oversampling if desired
        if use_feature_augment:
            X_train, y_train = feature_augment(X_train, y_train, noise_std=0.01, random_state=RANDOM_STATE)
            print("After feature-level augmentation:", np.bincount(y_train))
        # also apply NumPy oversampler to ensure exact balance
        X_train, y_train = random_oversample(X_train, y_train, random_state=RANDOM_STATE)
        print("Class counts after oversampling:", np.bincount(y_train))

        # Weighted loss for class imbalance
        classes = np.unique(y_train)
        class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
        # compute_class_weight returns weight per class in sorted(classes)
        # build weight vector for CrossEntropyLoss that expects weight for each class index 0..C-1
        full_weights = np.ones(np.max(classes)+1, dtype=float)
        for cls, w in zip(classes, class_weights):
            full_weights[cls] = w
        class_weights_torch = torch.tensor(full_weights, dtype=torch.float).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_torch)

        # train/val split from training data
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE)

        train_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                                torch.tensor(y_tr, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                                              torch.tensor(y_val, dtype=torch.long)), batch_size=BATCH_SIZE)
        test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                               torch.tensor(y_test, dtype=torch.long)), batch_size=BATCH_SIZE)

        model = DNNModel(X_train.shape[1], n_classes=int(np.max(y_train)+1)).to(device)
        optimizer = optim.Adam(model.parameters(), lr=LR)

        print("🚀 Training on GPU with class-weighted loss + augmentation + oversampling...")
        train_one_fold(model, train_loader, val_loader, criterion, optimizer, epochs=EPOCHS)

        # Evaluate on test set
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb = Xb.to(device)
                out = model(Xb)
                preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                trues.extend(yb.numpy())

        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average="macro")
        print(f"✅ Fold {fold}: Accuracy={acc:.4f}, F1-macro={f1:.4f}")
        print(classification_report(trues, preds, digits=3))
        conf = confusion_matrix(trues, preds)
        print("Confusion Matrix:\n", conf)

        fold_accs.append(acc)
        fold_f1s.append(f1)
        if total_conf_mat is None:
            total_conf_mat = conf
        else:
            # align shapes (in case some fold lacks classes)
            max_dim = max(total_conf_mat.shape[0], conf.shape[0])
            # expand matrices to max_dim x max_dim
            def expand_cm(cm, size):
                new = np.zeros((size, size), dtype=int)
                new[:cm.shape[0], :cm.shape[1]] = cm
                return new
            total_conf_mat = expand_cm(total_conf_mat, max_dim) + expand_cm(conf, max_dim)

    # final print
    print("\n🔥 FINAL RESULTS 🔥")
    if fold_accs:
        print(f"Mean Accuracy: {np.mean(fold_accs):.4f}")
        print(f"Mean F1-macro: {np.mean(fold_f1s):.4f}")
    if total_conf_mat is not None:
        print("Aggregated Confusion Matrix (summed across folds):\n", total_conf_mat)

# -------------------------------
# 14️⃣ Run pipeline
# -------------------------------
entries = build_dataset(DATASET_DIR, id_to_label)
evaluate_pipeline(entries, Nw=Nw_default, n_folds=3, use_feature_augment=True, tile_mel_to_segments=True)


✅ Device detected: cuda
✅ Found 2176 utterances from 272 subjects.

===== Fold 1/3 =====


Extracting features (fold 1): 100%|█████████████████████████████| 2176/2176 [03:32<00:00, 10.23it/s]


Raw sizes -> (184487, 259) (184487,) (100591, 259) (100591,)
After feature-level augmentation: [77065 77065 77065 77065 77065]
Class counts after oversampling: [77065 77065 77065 77065 77065]
🚀 Training on GPU with class-weighted loss + augmentation + oversampling...
Epoch 1/40 | TrainLoss=0.2848 | ValLoss=0.0390
Epoch 2/40 | TrainLoss=0.0832 | ValLoss=0.0097
Epoch 3/40 | TrainLoss=0.0551 | ValLoss=0.0064
Epoch 4/40 | TrainLoss=0.0421 | ValLoss=0.0041
Epoch 5/40 | TrainLoss=0.0344 | ValLoss=0.0033
Epoch 6/40 | TrainLoss=0.0299 | ValLoss=0.0039
Epoch 7/40 | TrainLoss=0.0267 | ValLoss=0.0025
Epoch 8/40 | TrainLoss=0.0245 | ValLoss=0.0026
Epoch 9/40 | TrainLoss=0.0219 | ValLoss=0.0020
Epoch 10/40 | TrainLoss=0.0206 | ValLoss=0.0017
Epoch 11/40 | TrainLoss=0.0184 | ValLoss=0.0013
Epoch 12/40 | TrainLoss=0.0173 | ValLoss=0.0014
Epoch 13/40 | TrainLoss=0.0164 | ValLoss=0.0014
Epoch 14/40 | TrainLoss=0.0156 | ValLoss=0.0009
Epoch 15/40 | TrainLoss=0.0147 | ValLoss=0.0012
Epoch 16/40 | TrainLo

Extracting features (fold 2): 100%|█████████████████████████████| 2176/2176 [03:26<00:00, 10.54it/s]


Raw sizes -> (189532, 259) (189532,) (95546, 259) (95546,)
After feature-level augmentation: [79820 79820 79820 79820 79820]
Class counts after oversampling: [79820 79820 79820 79820 79820]
🚀 Training on GPU with class-weighted loss + augmentation + oversampling...
Epoch 1/40 | TrainLoss=0.2759 | ValLoss=0.0293
Epoch 2/40 | TrainLoss=0.0743 | ValLoss=0.0055
Epoch 3/40 | TrainLoss=0.0479 | ValLoss=0.0036
Epoch 4/40 | TrainLoss=0.0377 | ValLoss=0.0021
Epoch 5/40 | TrainLoss=0.0309 | ValLoss=0.0016
Epoch 6/40 | TrainLoss=0.0260 | ValLoss=0.0009
Epoch 7/40 | TrainLoss=0.0231 | ValLoss=0.0007
Epoch 8/40 | TrainLoss=0.0212 | ValLoss=0.0011
Epoch 9/40 | TrainLoss=0.0190 | ValLoss=0.0005
Epoch 10/40 | TrainLoss=0.0177 | ValLoss=0.0004
Epoch 11/40 | TrainLoss=0.0159 | ValLoss=0.0002
Epoch 12/40 | TrainLoss=0.0144 | ValLoss=0.0004
Epoch 13/40 | TrainLoss=0.0141 | ValLoss=0.0003
Epoch 14/40 | TrainLoss=0.0138 | ValLoss=0.0002
Epoch 15/40 | TrainLoss=0.0121 | ValLoss=0.0003
Epoch 16/40 | TrainLoss

Extracting features (fold 3): 100%|█████████████████████████████| 2176/2176 [03:28<00:00, 10.43it/s]


Raw sizes -> (196137, 259) (196137,) (88941, 259) (88941,)
After feature-level augmentation: [86585 86585 86585 86585 86585]
Class counts after oversampling: [86585 86585 86585 86585 86585]
🚀 Training on GPU with class-weighted loss + augmentation + oversampling...
Epoch 1/40 | TrainLoss=0.2896 | ValLoss=0.0370
Epoch 2/40 | TrainLoss=0.0837 | ValLoss=0.0127
Epoch 3/40 | TrainLoss=0.0564 | ValLoss=0.0074
Epoch 4/40 | TrainLoss=0.0431 | ValLoss=0.0038
Epoch 5/40 | TrainLoss=0.0355 | ValLoss=0.0032
Epoch 6/40 | TrainLoss=0.0304 | ValLoss=0.0031
Epoch 7/40 | TrainLoss=0.0277 | ValLoss=0.0025
Epoch 8/40 | TrainLoss=0.0245 | ValLoss=0.0021
Epoch 9/40 | TrainLoss=0.0222 | ValLoss=0.0019
Epoch 10/40 | TrainLoss=0.0210 | ValLoss=0.0017
Epoch 11/40 | TrainLoss=0.0190 | ValLoss=0.0016
Epoch 12/40 | TrainLoss=0.0178 | ValLoss=0.0012
Epoch 13/40 | TrainLoss=0.0165 | ValLoss=0.0016
Epoch 14/40 | TrainLoss=0.0165 | ValLoss=0.0017
Epoch 15/40 | TrainLoss=0.0148 | ValLoss=0.0018
Epoch 16/40 | TrainLoss

In [3]:
# Complete Kaggle-ready pipeline: audio-only spectral + MFCC suprasegmental features + GPU DNN
# Run this cell in a Kaggle notebook with GPU enabled.
# Requirements (Kaggle typically has these): numpy, pandas, librosa, soundfile, tqdm, scikit-learn, torch

import os
import time
import math
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# -------------------------
# CONFIG (edit paths if needed)
# -------------------------
DATASET_DIR = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training"
METADATA_PATH = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx"

SR_TARGET = 16000
FRAME_WIN = 0.020   # seconds (suprasegment window baseline)
FRAME_HOP = 0.010   # seconds (frame hop for MFCC)
Nw_default = 0.8
Nsh = 0.1
RANDOM_STATE = 42

BATCH_SIZE = 256
EPOCHS = 40
LR = 1e-3
PATIENCE = 6       # early stopping patience on validation loss
N_FOLDS = 3        # subject-level CV folds (set 5 if you want)
SUBSAMPLE_TRAIN_WINDOWS = None  # set int to limit training windows for faster runs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("Warning: GPU not detected. The code is written to use GPU; enable GPU in Kaggle settings.")

# -------------------------
# Helpers: metadata + dataset build
# -------------------------
def read_metadata(metadata_path):
    df = pd.read_excel(metadata_path)
    df.columns = [c.strip().lower() for c in df.columns]
    id_col = next((c for c in df.columns if 'id' in c), None)
    class_col = next((c for c in df.columns if 'class' in c), None)
    if id_col is None or class_col is None:
        raise ValueError("Metadata Excel must contain ID and Class columns")
    # normalize id strings
    mapping = {}
    for _, r in df.iterrows():
        key = str(r[id_col]).strip().upper()
        cls = int(r[class_col])
        mapping[key] = {'label': cls}
    return mapping

def build_entries(root_dir, id_to_info):
    """
    Walk the training folder and return list of utterance entries:
    [{'subject_id': 'ID123', 'utterance_path': '/.../ID123_phonationA.wav', 'label': int}]
    """
    entries = []
    for root, dirs, files in os.walk(root_dir):
        for f in sorted(files):
            if not f.lower().endswith('.wav'):
                continue
            base = os.path.splitext(f)[0].strip().upper()
            subj = base.split('_')[0]
            subj_clean = subj.replace('-', '').replace(' ', '')
            # find matching id in metadata with tolerant normalization
            match = None
            for k in id_to_info.keys():
                if k.replace('-', '').replace(' ', '') == subj_clean:
                    match = k; break
            if match is None:
                # skip if not found
                continue
            entries.append({
                'subject_id': match,
                'utterance_path': os.path.join(root, f),
                'label': id_to_info[match]['label']
            })
    print(f"Found {len(entries)} utterances from {len(set(e['subject_id'] for e in entries))} subjects.")
    return entries

# -------------------------
# Feature extraction utilities
# -------------------------
def extract_spectral_features_from_audio(x, sr):
    """
    Return a compact vector of spectral statistics (shape ~35-45).
    Uses STFT-based features + simple statistics across frames.
    """
    # STFT parameters
    n_fft = 1024
    hop_length = 256

    S = np.abs(librosa.stft(x, n_fft=n_fft, hop_length=hop_length, center=True))
    S_power = S**2 + 1e-10

    # spectral features per frame
    centroid = librosa.feature.spectral_centroid(S=S)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=S)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]
    zcr = librosa.feature.zero_crossing_rate(x, hop_length=hop_length)[0]
    rms = librosa.feature.rms(S=S)[0]

    # spectral entropy (per frame)
    p = S_power / (np.sum(S_power, axis=0, keepdims=True) + 1e-12)
    entropy = -np.sum(p * np.log2(p + 1e-12), axis=0)
    # normalize entropy by log(#bins)
    entropy = entropy / math.log2(S.shape[0] + 1e-12)

    def stats(v):
        return np.array([np.mean(v), np.std(v), np.median(v), np.percentile(v, 25), np.percentile(v,75)])

    vect = np.hstack([
        stats(centroid),
        stats(bandwidth),
        stats(rolloff),
        stats(flatness),
        stats(zcr),
        stats(rms),
        stats(entropy)
    ])
    return vect  # length ~ 7*5 = 35

def compute_mfcc36_frames(x, sr):
    hop_length = int(round(FRAME_HOP * sr))
    win_length = int(round(FRAME_WIN * sr))
    n_fft = max(512, win_length * 2)
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13, n_fft=n_fft, hop_length=hop_length)
    if mfcc.shape[0] < 13:
        # fallback (shouldn't really happen)
        mfcc = np.pad(mfcc, ((0,13-mfcc.shape[0]),(0,0)), mode='constant')
    mfcc12 = mfcc[1:13, :]
    delta = librosa.feature.delta(mfcc12)
    delta2 = librosa.feature.delta(mfcc12, order=2)
    feat = np.concatenate([mfcc12, delta, delta2], axis=0)  # (36, n_frames)
    return feat.T  # (n_frames,36)

def suprasegmental_from_frames(frames, Nw=Nw_default, Nsh=Nsh):
    """
    frames: (n_frames, feat_dim) e.g. (T,36)
    returns (n_windows, feat_dim*3) where each window => mean, median, std concatenated
    """
    hop_s = FRAME_HOP
    win_s = FRAME_WIN
    if Nw <= win_s:
        frames_per_window = 1
    else:
        frames_per_window = int(round((Nw - win_s) / hop_s)) + 1
    shift_per_window = max(1, int(round(Nsh / hop_s)))
    n_frames = frames.shape[0]
    supraseg_feats = []
    start = 0
    while start + frames_per_window <= n_frames:
        win_frames = frames[start:start + frames_per_window, :]
        mean = np.mean(win_frames, axis=0)
        median = np.median(win_frames, axis=0)
        sd = np.std(win_frames, axis=0)
        supraseg_feats.append(np.concatenate([mean, median, sd], axis=0))
        start += shift_per_window
    if len(supraseg_feats) == 0:
        mean = np.mean(frames, axis=0)
        median = np.median(frames, axis=0)
        sd = np.std(frames, axis=0)
        supraseg_feats.append(np.concatenate([mean, median, sd], axis=0))
    return np.array(supraseg_feats)  # (n_windows, 36*3=108)

def extract_features_from_wav(path, sr_target=SR_TARGET, Nw=Nw_default):
    """
    Returns: numpy array shape (n_windows, feat_dim)
      feat_dim = spectral_feat_dim + supraseg_dim (35 + 108 = 143 by default)
    """
    try:
        x, orig_sr = sf.read(path)
    except Exception as e:
        raise RuntimeError(f"Failed to read {path}: {e}")
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    if orig_sr != sr_target:
        x = librosa.resample(y=x.astype(np.float32), orig_sr=orig_sr, target_sr=sr_target)
    x = x.astype(np.float32)

    # spectral stats (global across utterance)
    spec = extract_spectral_features_from_audio(x, sr_target)  # ~35-d

    # mfcc frames -> supraseg windows (per-window 108)
    mfcc_frames = compute_mfcc36_frames(x, sr_target)  # (T,36)
    mfcc_frames = (mfcc_frames - np.mean(mfcc_frames, axis=0)) / (np.std(mfcc_frames, axis=0) + 1e-12)
    supr = suprasegmental_from_frames(mfcc_frames, Nw=Nw, Nsh=Nsh)  # (n_windows,108)

    # concatenate spec to each supr window
    combined = np.hstack([np.tile(spec.reshape(1, -1), (supr.shape[0], 1)), supr])
    # shape (n_windows, spec_dim + 108)
    return combined

# -------------------------
# Cache all features once (so feature extraction done only once)
# -------------------------
def build_feature_cache(entries, cache_progress=True):
    features_by_entry = {}
    errors = []
    seq = entries
    if cache_progress:
        seq = tqdm(entries, desc="Extracting all features", unit="file")
    for e in seq:
        p = e['utterance_path']
        try:
            feats = extract_features_from_wav(p, Nw=Nw_default)
            features_by_entry[p] = feats  # ndarray (n_windows, feat_dim)
        except Exception as ex:
            errors.append((p, str(ex)))
    if errors:
        print(f"Warnings: failed to extract {len(errors)} files (first 3):", errors[:3])
    return features_by_entry

# -------------------------
# Oversampling (NumPy) to balance classes
# -------------------------
def random_oversample(X, y, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y, return_counts=True)
    max_count = counts.max()
    X_res_list = []
    y_res_list = []
    for cls, cnt in zip(classes, counts):
        mask = (y == cls)
        Xc = X[mask]
        yc = y[mask]
        if cnt < max_count:
            idx = rng.choice(len(Xc), max_count - cnt, replace=True)
            X_extra = Xc[idx]
            y_extra = np.full(len(idx), cls)
            Xc = np.vstack([Xc, X_extra])
            yc = np.hstack([yc, y_extra])
        X_res_list.append(Xc)
        y_res_list.append(yc)
    X_res = np.vstack(X_res_list)
    y_res = np.hstack(y_res_list)
    return X_res, y_res

# -------------------------
# PyTorch model
# -------------------------
class DNNModel(nn.Module):
    def __init__(self, input_dim, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.35),
            nn.Linear(256, n_classes)
        )
    def forward(self, x):
        return self.net(x)

# -------------------------
# Training helpers
# -------------------------
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for Xb, yb in loader:
        Xb = Xb.to(device); yb = yb.to(device)
        optimizer.zero_grad()
        out = model(Xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
    return total_loss / len(loader.dataset)

def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    preds = []
    trues = []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device); yb = yb.to(device)
            out = model(Xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * Xb.size(0)
            preds_batch = torch.argmax(out, dim=1).cpu().numpy()
            preds.extend(preds_batch)
            trues.extend(yb.cpu().numpy())
    return total_loss / len(loader.dataset), np.array(preds), np.array(trues)

# -------------------------
# Full subject-level CV pipeline
# -------------------------
def run_subject_cv(entries, features_cache, n_folds=N_FOLDS, subsample_train_windows=SUBSAMPLE_TRAIN_WINDOWS):
    # subjects and labels
    subjects = np.array(sorted(set(e['subject_id'] for e in entries)))
    subj_to_label = {s: next(e['label'] for e in entries if e['subject_id'] == s) - 1 for s in subjects}  # zero-index labels
    labels = np.array([subj_to_label[s] for s in subjects])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    fold_results = []
    feat_dim = None

    for fold, (train_idx, test_idx) in enumerate(skf.split(subjects, labels), 1):
        print("\n" + "="*40)
        print(f"FOLD {fold}/{n_folds}")
        train_subj = set(subjects[train_idx])
        test_subj = set(subjects[test_idx])

        # assemble training/test windows from cached features
        X_train_windows = []
        y_train_windows = []
        X_test_windows = []
        y_test_windows = []

        # track mapping for subject-level majority vote later
        test_subject_windows = defaultdict(list)

        # loop entries (fast because features_cache holds arrays)
        for e in tqdm(entries, desc=f"Assembling windows for fold {fold}", unit="file"):
            p = e['utterance_path']
            subj = e['subject_id']
            if p not in features_cache:
                continue
            feats = features_cache[p]    # (n_win, feat_dim)
            if feat_dim is None:
                feat_dim = feats.shape[1]
            lab = e['label'] - 1
            if subj in train_subj:
                X_train_windows.append(feats)
                y_train_windows.append(np.full(feats.shape[0], lab, dtype=int))
            elif subj in test_subj:
                X_test_windows.append(feats)
                y_test_windows.append(np.full(feats.shape[0], lab, dtype=int))
                # store mapping for subject-level later: keep indices by subject
                # We'll record predicted windows per subject using test_subject_windows[subj]
                # (we only need the windows for voting after prediction)
                test_subject_windows[subj].append(feats)

        if len(X_train_windows) == 0 or len(X_test_windows) == 0:
            print("No windows found for this fold - skipping.")
            continue

        X_train = np.vstack(X_train_windows)
        y_train = np.hstack(y_train_windows)
        X_test = np.vstack(X_test_windows)
        y_test = np.hstack(y_test_windows)
        print(f"Windows: Train {X_train.shape}, Test {X_test.shape}")

        # optional subsample training windows (speed)
        if subsample_train_windows is not None and X_train.shape[0] > subsample_train_windows:
            idx = np.random.RandomState(RANDOM_STATE).choice(X_train.shape[0], subsample_train_windows, replace=False)
            X_train = X_train[idx]
            y_train = y_train[idx]
            print(f"Subsampled training windows to {subsample_train_windows}")

        # scale features (fit on train windows)
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

        # oversample training windows to balance classes
        X_train_bal, y_train_bal = random_oversample(X_train, y_train, random_state=RANDOM_STATE)
        print("Class counts after oversampling:", np.bincount(y_train_bal))

        # train/val split
        X_tr, X_val, y_tr, y_val = train_test_split(X_train_bal, y_train_bal, test_size=0.15,
                                                    stratify=y_train_bal, random_state=RANDOM_STATE)

        # build PyTorch loaders
        train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long))
        val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
        test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

        n_classes = len(np.unique(y_train_bal))
        model = DNNModel(input_dim if feat_dim is None else feat_dim, n_classes=n_classes).to(device)

        # class weights for loss (balanced)
        classes = np.unique(y_tr)
        class_weights = compute_class_weight('balanced', classes=np.arange(n_classes), y=y_tr)
        class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(model.parameters(), lr=LR)

        # training loop with early stopping on val loss
        best_val_loss = float('inf')
        best_epoch = -1
        best_state = None
        epochs_no_improve = 0

        for epoch in range(1, EPOCHS+1):
            t0 = time.time()
            train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
            val_loss, val_preds, val_trues = eval_model(model, val_loader, criterion, device)
            t1 = time.time()
            if val_loss < best_val_loss - 1e-6:
                best_val_loss = val_loss
                best_state = {k: v.cpu() for k,v in model.state_dict().items()}
                best_epoch = epoch
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            print(f"Epoch {epoch}/{EPOCHS} | TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | time={(t1-t0):.1f}s")
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping (no improvement for {PATIENCE} epochs)")
                break

        # restore best model
        if best_state is not None:
            model.load_state_dict(best_state)

        # Evaluate on test windows (window-level)
        test_loss, test_preds, test_trues = eval_model(model, test_loader, criterion, device)
        win_acc = accuracy_score(test_trues, test_preds)
        win_f1 = f1_score(test_trues, test_preds, average='macro')
        print(f"Window-level Test: Acc={win_acc:.4f}, F1-macro={win_f1:.4f}")

        # Subject-level majority vote
        subj_preds = []
        subj_trues = []
        for subj in sorted(test_subj):
            # gather all windows for this subject from features_cache (may come from multiple utterances)
            subj_windows = []
            for feats in test_subject_windows[subj]:
                # feats are raw (not scaled); need to transform and then scale
                subj_windows.append(scaler.transform(feats))
            if len(subj_windows) == 0:
                continue
            subj_windows = np.vstack(subj_windows)
            # create loader and predict
            model.eval()
            preds_subj = []
            with torch.no_grad():
                # batch predict
                for i in range(0, subj_windows.shape[0], BATCH_SIZE):
                    Xb = torch.tensor(subj_windows[i:i+BATCH_SIZE], dtype=torch.float32).to(device)
                    out = model(Xb)
                    preds_subj.extend(torch.argmax(out, dim=1).cpu().numpy())
            # majority vote
            maj = Counter(preds_subj).most_common(1)[0][0]
            subj_preds.append(maj)
            subj_trues.append(subj_to_label[subj])

        subj_acc = accuracy_score(subj_trues, subj_preds) if len(subj_trues)>0 else float('nan')
        subj_f1 = f1_score(subj_trues, subj_preds, average='macro') if len(subj_trues)>0 else float('nan')
        print(f"Subject-level Test: Acc={subj_acc:.4f}, F1-macro={subj_f1:.4f}")
        print("Classification report (subject-level):")
        print(classification_report(subj_trues, subj_preds, digits=3))

        fold_results.append({
            'fold': fold,
            'win_acc': win_acc,
            'win_f1': win_f1,
            'subj_acc': subj_acc,
            'subj_f1': subj_f1,
            'n_train_windows': X_tr.shape[0],
            'n_val_windows': X_val.shape[0],
            'n_test_windows': X_test.shape[0]
        })

    # summary
    print("\n==== SUMMARY across folds ====")
    if len(fold_results) == 0:
        print("No folds completed.")
        return
    win_accs = [r['win_acc'] for r in fold_results]
    win_f1s = [r['win_f1'] for r in fold_results]
    subj_accs = [r['subj_acc'] for r in fold_results if not math.isnan(r['subj_acc'])]
    subj_f1s = [r['subj_f1'] for r in fold_results if not math.isnan(r['subj_f1'])]

    print(f"Window-level mean Acc: {np.mean(win_accs):.4f}, mean F1-macro: {np.mean(win_f1s):.4f}")
    if subj_accs:
        print(f"Subject-level mean Acc: {np.mean(subj_accs):.4f}, mean F1-macro: {np.mean(subj_f1s):.4f}")

    return fold_results

# -------------------------
# MAIN execution
# -------------------------
if __name__ == "__main__":
    print("Reading metadata...")
    id_to_info = read_metadata(METADATA_PATH)

    print("Building entries list (matching WAVs to metadata)...")
    entries = build_entries(DATASET_DIR, id_to_info)

    # Precompute features for all utterances (this is the slow part but done once)
    start_all = time.time()
    features_cache = build_feature_cache(entries, cache_progress=True)
    print(f"Feature extraction finished in {(time.time()-start_all)/60:.2f} minutes. Cached entries: {len(features_cache)}")

    # Run subject-level CV (training + evaluation)
    results = run_subject_cv(entries, features_cache, n_folds=N_FOLDS, subsample_train_windows=SUBSAMPLE_TRAIN_WINDOWS)


Device: cuda
Reading metadata...
Building entries list (matching WAVs to metadata)...
Found 2176 utterances from 272 subjects.


Extracting all features: 100%|██████████| 2176/2176 [03:01<00:00, 11.96file/s]


Warnings: failed to extract 2176 files (first 3): [('/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training/phonationE/ID000_phonationE.wav', 'Since S.shape[-2] is 513, frame_length is expected to be 1024 or 1025; found 2048'), ('/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training/phonationE/ID001_phonationE.wav', 'Since S.shape[-2] is 513, frame_length is expected to be 1024 or 1025; found 2048'), ('/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training/phonationE/ID002_phonationE.wav', 'Since S.shape[-2] is 513, frame_length is expected to be 1024 or 1025; found 2048')]
Feature extraction finished in 3.03 minutes. Cached entries: 0

FOLD 1/3


Assembling windows for fold 1: 100%|██████████| 2176/2176 [00:00<00:00, 2086603.91file/s]


No windows found for this fold - skipping.

FOLD 2/3


Assembling windows for fold 2: 100%|██████████| 2176/2176 [00:00<00:00, 3227300.39file/s]


No windows found for this fold - skipping.

FOLD 3/3


Assembling windows for fold 3: 100%|██████████| 2176/2176 [00:00<00:00, 2222797.25file/s]

No windows found for this fold - skipping.

==== SUMMARY across folds ====
No folds completed.


In [2]:
# ===============================================================
# 🔥 ALS Severity Classification (Combined MFCC + Spectral Features, GPU DNN)
# ===============================================================
import os
import numpy as np
import pandas as pd
import librosa, soundfile as sf
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import skew, kurtosis
from scipy.fftpack import dct

# -------------------------------
# 1️⃣ GPU setup
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device detected: {device}")
if device.type != "cuda":
    raise SystemError("❌ GPU not detected. Please enable GPU in Kaggle runtime settings.")

# -------------------------------
# 2️⃣ Configuration
# -------------------------------
SR_TARGET = 16000
FRAME_WIN = 0.020
FRAME_HOP = 0.010
Nw_default = 0.8
Nsh = 0.1
BATCH_SIZE = 128
EPOCHS = 40
LR = 1e-3
RANDOM_STATE = 42
DATASET_DIR = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/training"
META_PATH = "/kaggle/input/sandspgc/SAND_Challenge_task1_dataset (2)/task1/sand_task_1.xlsx"
MEL_DIR = "/kaggle/input/melspectograms/mel_spectrograms"

# -------------------------------
# 3️⃣ Load Audio + Compute MFCC suprasegmental
# -------------------------------
def load_audio(filename, sr=SR_TARGET):
    x, orig_sr = sf.read(filename)
    if x.ndim > 1: 
        x = np.mean(x, axis=1)
    if orig_sr != sr:
        x = librosa.resample(y=x.astype(np.float32), orig_sr=orig_sr, target_sr=sr)
    return x.astype(np.float32), sr

def compute_mfcc_36(x, sr=SR_TARGET):
    hop = int(round(FRAME_HOP * sr))
    win = int(round(FRAME_WIN * sr))
    mfcc = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13, n_fft=max(512, win * 2), hop_length=hop)
    mfcc12 = mfcc[1:13, :]
    delta = librosa.feature.delta(mfcc12)
    delta2 = librosa.feature.delta(mfcc12, order=2)
    return np.concatenate([mfcc12, delta, delta2], axis=0).T

def apply_cmvn(frames):
    mu, sigma = np.mean(frames, 0), np.std(frames, 0)
    sigma[sigma == 0] = 1.0
    return (frames - mu) / sigma

def suprasegmental_from_frames(frames, Nw=Nw_default, Nsh=Nsh):
    hop_s, win_s = FRAME_HOP, FRAME_WIN
    frames_per_window = int(round((Nw - win_s) / hop_s)) + 1 if Nw > win_s else 1
    shift = int(round(Nsh / hop_s))
    supraseg_feats, start = [], 0
    while start + frames_per_window <= frames.shape[0]:
        w = frames[start:start + frames_per_window]
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)],0))
        start += shift
    if not supraseg_feats:
        w = frames
        supraseg_feats.append(np.concatenate([np.mean(w,0), np.median(w,0), np.std(w,0)],0))
    return np.array(supraseg_feats)

def extract_mfcc_supra(utt_path, Nw=Nw_default):
    x, sr = load_audio(utt_path)
    frames = compute_mfcc_36(x, sr)
    frames = apply_cmvn(frames)
    return suprasegmental_from_frames(frames, Nw, Nsh)

# -------------------------------
# 4️⃣ Handcrafted Spectral Features (from PNG)
# -------------------------------
def extract_mel_features(mel):
    eps = 1e-10
    mel = np.maximum(np.asarray(mel, dtype=float), 0.0)
    if mel.ndim != 2:
        return np.zeros(1)
    n_mels, n_frames = mel.shape
    freqs = np.arange(n_mels, dtype=float) + 1.0
    mag = mel + eps

    centroid = (freqs[:, None] * mag).sum(axis=0) / (mag.sum(axis=0) + eps)
    bandwidth = np.sqrt(((freqs[:, None] - centroid[None, :])**2 * mag).sum(axis=0) / (mag.sum(axis=0) + eps))
    cumsum = np.cumsum(mag, axis=0)
    total_energy = cumsum[-1, :] + eps
    rolloff = np.array([np.searchsorted(cumsum[:, t], 0.85 * total_energy[t]) for t in range(n_frames)])
    geo_mean = np.exp(np.mean(np.log(mag), axis=0))
    arith_mean = np.mean(mag, axis=0)
    flatness = geo_mean / (arith_mean + eps)
    flux = np.sqrt(np.diff(mag / (arith_mean[None, :] + eps), axis=1)**2).sum(axis=0)
    flux = np.concatenate(([0.0], flux))
    p = mag / (mag.sum(axis=0, keepdims=True) + eps)
    entropy = - (p * np.log2(p + eps)).sum(axis=0) / np.log2(n_mels + eps)
    n1, n2 = n_mels // 3, 2 * n_mels // 3
    low_energy = mag[:n1, :].sum(axis=0)
    mid_energy = mag[n1:n2, :].sum(axis=0)
    high_energy = mag[n2:, :].sum(axis=0)
    log_mel = np.log(mag)
    mfccs = dct(log_mel, type=2, axis=0, norm="ortho")[:13, :]

    def stats_vec(v):
        return np.array([
            np.nanmean(v), np.nanstd(v),
            skew(v, bias=False) if v.size > 2 else 0.0,
            kurtosis(v, bias=False) if v.size > 3 else 0.0,
            np.nanmedian(v),
            np.nanpercentile(v, 25.0),
            np.nanpercentile(v, 75.0)
        ])
    feats = []
    for vec in [centroid, bandwidth, rolloff, flatness, flux, entropy, low_energy, mid_energy, high_energy]:
        feats.append(stats_vec(vec))
    feats.append(np.hstack([np.mean(mfccs, axis=1), np.std(mfccs, axis=1)]))
    feats.append(np.array([np.nanmean(mag), np.nanstd(mag), np.nanmax(mag), np.nanmin(mag)]))
    return np.nan_to_num(np.hstack(feats))

def extract_png_features(subj_id):
    """Finds a mel-spectrogram PNG for this subject and extracts features."""
    for root, _, files in os.walk(MEL_DIR):
        for f in files:
            if f.lower().endswith(".png") and subj_id in f:
                try:
                    img = Image.open(os.path.join(root, f)).convert("L")
                    mel = np.array(img)
                    return extract_mel_features(mel)
                except:
                    continue
    return np.zeros(150)  # fallback if no PNG found

# -------------------------------
# 5️⃣ Dataset builder (Audio + PNG)
# -------------------------------
def build_dataset(root_dir, metadata_path):
    meta = pd.read_excel(metadata_path)
    meta.columns = [c.strip().lower() for c in meta.columns]
    id_col = next(c for c in meta.columns if "id" in c)
    class_col = next(c for c in meta.columns if "class" in c)
    id_to_label = {str(r[id_col]).strip().upper(): int(r[class_col]) - 1 for _, r in meta.iterrows()}

    entries = []
    for folder, _, files in os.walk(root_dir):
        for f in files:
            if not f.endswith(".wav"): continue
            subj = f.split("_")[0].strip().upper()
            subj_clean = subj.replace("-", "").replace(" ", "")
            match = next((k for k in id_to_label if k.replace("-", "").replace(" ", "") == subj_clean), None)
            if match:
                entries.append({
                    "subject_id": match,
                    "utterance_path": os.path.join(folder, f),
                    "label": id_to_label[match]
                })
    print(f"✅ Found {len(entries)} utterances from {len(set(e['subject_id'] for e in entries))} subjects.")
    return entries

# -------------------------------
# 6️⃣ Oversampling
# -------------------------------
def random_oversample(X, y, random_state=42):
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y, return_counts=True)
    max_count = counts.max()
    X_res, y_res = [], []
    for cls, count in zip(classes, counts):
        mask = y == cls
        X_cls = X[mask]
        y_cls = y[mask]
        n_needed = max_count - count
        if n_needed > 0:
            idx = rng.choice(len(X_cls), n_needed, replace=True)
            X_cls = np.vstack([X_cls, X_cls[idx]])
            y_cls = np.hstack([y_cls, np.full(n_needed, cls)])
        X_res.append(X_cls)
        y_res.append(y_cls)
    return np.vstack(X_res), np.hstack(y_res)

# -------------------------------
# 7️⃣ DNN Model
# -------------------------------
class DNNModel(nn.Module):
    def __init__(self, input_dim, n_classes=5):
        super(DNNModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )
    def forward(self, x):
        return self.net(x)

# -------------------------------
# 8️⃣ Train/Evaluate Combined
# -------------------------------
def evaluate_pipeline(entries, Nw=0.8, n_folds=3):
    subjects = np.array(sorted(set(e["subject_id"] for e in entries)))
    labels = np.array([next(e["label"] for e in entries if e["subject_id"] == s) for s in subjects])
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    accs, f1s = [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(subjects, labels), 1):
        print(f"\n===== Fold {fold}/{n_folds} =====")
        train_subj, test_subj = set(subjects[train_idx]), set(subjects[test_idx])

        X_train, y_train, X_test, y_test = [], [], [], []
        for e in tqdm(entries, desc=f"Extracting features (fold {fold})", ncols=100):
            try:
                mfcc_feat = extract_mfcc_supra(e["utterance_path"], Nw)
                png_feat = extract_png_features(e["subject_id"])
                combined = np.hstack([np.mean(mfcc_feat, axis=0), png_feat])  # combine per utterance
                if e["subject_id"] in train_subj:
                    X_train.append(combined); y_train.append(e["label"])
                elif e["subject_id"] in test_subj:
                    X_test.append(combined); y_test.append(e["label"])
            except Exception:
                continue

        X_train, y_train, X_test, y_test = map(np.array, (X_train, y_train, X_test, y_test))
        sc = StandardScaler().fit(X_train)
        X_train, X_test = sc.transform(X_train), sc.transform(X_test)

        X_train, y_train = random_oversample(X_train, y_train, random_state=RANDOM_STATE)
        print("Class counts after oversampling:", np.bincount(y_train))

        class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE)
        train_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                                torch.tensor(y_tr, dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                                              torch.tensor(y_val, dtype=torch.long)), batch_size=BATCH_SIZE)
        test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                               torch.tensor(y_test, dtype=torch.long)), batch_size=BATCH_SIZE)

        model = DNNModel(X_train.shape[1]).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(model.parameters(), lr=LR)

        print("🚀 Training on GPU...")
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                optimizer.zero_grad()
                out = model(Xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            model.eval()
            with torch.no_grad():
                val_loss = np.mean([criterion(model(Xv.to(device)), yv.to(device)).item() for Xv, yv in val_loader])
            print(f"Epoch {epoch+1}/{EPOCHS} | TrainLoss={total_loss/len(train_loader):.4f} | ValLoss={val_loss:.4f}")

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb = Xb.to(device)
                out = model(Xb)
                preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                trues.extend(yb.numpy())

        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds, average="macro")
        print(f"✅ Fold {fold}: Accuracy={acc:.4f}, F1-macro={f1:.4f}")
        accs.append(acc); f1s.append(f1)

    print("\n🔥 FINAL RESULTS 🔥")
    print(f"Mean Accuracy: {np.mean(accs):.4f}")
    print(f"Mean F1-macro: {np.mean(f1s):.4f}")

# -------------------------------
# 9️⃣ Run pipeline
# -------------------------------
entries = build_dataset(DATASET_DIR, META_PATH)
evaluate_pipeline(entries, Nw=0.8, n_folds=3)


✅ Device detected: cuda


KeyboardInterrupt: 